<a href="https://colab.research.google.com/github/dorothea-pudding/114-1_TAICA_homework/blob/main/9.%20AI%E4%BB%A3%E7%90%86%E8%A8%AD%E8%A8%88%E6%A8%A1%E5%BC%8F_Reflection%EF%BC%88%E9%99%84%E5%9C%96%E5%8A%9F%E8%83%BD%E6%9C%AA%E5%AE%8C%E6%88%90%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. Reflection 的任務設計

#### 🌟 任務說明：社群媒體發文優化小幫手

**🎯 流程說明：**
1. 使用者輸入今天想分享的內容
2. `model_writer` 生成第一版社群發文（風格活潑、有趣、使用 emoji、第一人稱）
3. `model_reviewer` 檢查內容是否夠生活化、通順、有趣，並提供具體修改建議
4. `model_writer` 根據建議產出第二版
5. Gradio 呈現：三個欄位：第一版、建議、第二版

#### 1. 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [ ]:
import os
from google.colab import userdata

Groq 可以用的模型請參考[這個連結](https://console.groq.com/docs/models)。

In [ ]:
#【使用 Mistral】
# api_key = userdata.get('Mistral')
# os.environ['MISTRAL']=api_key
# provider = "mistral"
# model = "ministral-8b-latest"

#【使用 OpenAI】
# api_key = userdata.get('OpenAI')
# os.environ['OPENAI_API_KEY']=api_key
# provider = "openai"
# model = "gpt-4o"

#【使用 Groq】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "openai/gpt-oss-120b"

In [ ]:
!pip install aisuite[all]

### 2. 基本的設定

In [ ]:
import aisuite as ai

In [ ]:
provider_writer = "groq"
model_writer="openai/gpt-oss-120b"

provider_reviewer = "groq"
model_reviewer = "openai/gpt-oss-120b"

#provider_reviewer = "openai"
#model_reviewer = "gpt-4o"

標準回應函式

In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="groq",
          model="openai/gpt-oss-120b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

讀圖片：現在的模型角色設定和生成的文本都沒有問題，但因為我的社群主題是遊戲攝影，所以想追加上傳圖片、根據圖片生成文本。

In [ ]:
# !pip install google-genai pillow

In [ ]:
"""
# Cell 3: 核心反射函式 (Reflect Agent) - 支援多模態

from PIL import Image # 確保在 Cell 2 或此處匯入

# 💡 函式簽名改變：新增了 image_path 參數
def reflect_post(text_input, image_path):
    # 1. 準備內容 (Content)

    contents = [text_input] # 初始內容只有文字
    image = None
    image_prompt = ""

    # **【圖片處理開始】**
    if image_path:
        try:
            # 使用 PIL 載入圖片
            image = Image.open(image_path)
            # 將圖片物件放在內容列表的第一位，以供模型參考
            contents.insert(0, image)

            image_prompt = "（重要：使用者提供了一張遊戲攝影圖片，請務必根據圖片內容和氛圍來撰寫貼文和提供建議。）"
        except Exception as e:
            # 圖片載入失敗的處理
            image_prompt = f"（圖片載入失敗，無法使用：{e}）"
            image_path = None
    else:
        image_prompt = "（未上傳圖片，請專注於文字內容的發想與改進）"
    # **【圖片處理結束】**


    # --- 🤖 代理人 1: 貼文撰寫者 (Model Writer) ---
    writer_system_prompt = f"""'''
    你是一位專業的社群貼文寫手，專門負責高品質的遊戲攝影貼文。
    ... (略，其餘 System Prompt 沿用原邏輯)
    {image_prompt}
    '''
"""


    # 呼叫 AI 時傳入 contents (可能包含圖片物件)
    response_1 = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=writer_system_prompt,
        ),
    )
    out1 = response_1.text

    # --- 🧠 代理人 2: 貼文審核者 (Model Reviewer) ---
    # 傳入內容時，同樣要檢查並包含圖片
    review_contents = [f"🌟 第一版貼文：\n{out1}"]
    if image:
        review_contents.insert(0, image)

    # ... (略，呼叫 AI 模型)

    # --- 🤖 代理人 3: 貼文改寫者 (Model Writer Rewrite) ---
    # 傳入內容時，同樣要檢查並包含圖片
    rewrite_contents = [
        f"🌟 第一版貼文：\n{out1}",
        f"🧐 修改建議：\n{out2}"
    ]
    if image:
        rewrite_contents.insert(0, image)

    # ... (略，呼叫 AI 模型)

    return out1, out2, out3
"""

####  3. 設定「作者」和「審查員」

In [ ]:
system_writer = """你是一位活潑、有趣的社群媒體幫手，
擅長幫我把遊戲攝影分享變得更吸睛、第一人稱風格、有情緒。
內容要聚焦在角色和畫面表現上。
如果沒有角色只有風景請聚焦在景色。如果沒有附圖就按提示生成文案
每篇貼文都要下標題，文章只要簡短，不要hashtag，
請用台灣習慣的中文回應。"""

system_reviewer = """你是一位文案潤稿專家，
擅長讓貼文更口語、生活化、自然有趣，請針對以下貼文給出具體修改建議。
文章只要簡短，不要hashtag
請用台灣習慣的中文回應。"""

In [ ]:
#combined_prompt = text_prompt + "\n\n" + image_prompt
def reflect_post(prompt):
    # Step 1: Writer 初稿
    first_version = reply(system_writer, prompt,
                          provider=provider_writer,
                          model=model_writer
                          )

    # Step 2: Reviewer 給建議
    suggestion = reply(system_reviewer, first_version,
                       provider=provider_reviewer,
                       model=model_reviewer
                       )

    # Step 3: Writer 再寫一次（根據建議）
    second_prompt = f"這是我剛剛寫的貼文：\n{first_version}\n\n這是修改建議：\n{suggestion}\n\n請根據這些建議，幫我改得更生活化、更自然。請用台灣習慣的中文, 並且只要輸出改好的文章就可以了。"
    second_version = reply(system_writer, second_prompt,
                           provider=provider_writer,
                           model=model_reviewer
                           )

    return first_version, suggestion, second_version

### 4. 用 Gradio 打造你的對話機器人 Web App!

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("### 🤖 社群貼文反思幫手（Reflection Agent）")
    user_input = gr.Textbox(label="請輸入你今天想分享的內容")
    btn = gr.Button("生成貼文 & 修正建議")

    with gr.Column():
        out1 = gr.Textbox(label="🌟 第一版貼文 (model_writer)", lines = 5)
        out2 = gr.Textbox(label="🧐 修改建議 (model_reviewer)", lines =5)
        out3 = gr.Textbox(label="✨ 第二版貼文 (model_writer 改寫)", lines = 5)

    btn.click(reflect_post, inputs=[user_input], outputs=[out1, out2, out3])

In [ ]:
demo.launch(share=True, debug=True)